# Upload CSV to DRP stable table

Uploads enriched CSV into stable DRP table `sbx_da.tmp_excel_3months_enriched_2026q1`.

Notebook uses robust flow: normalize values -> rename columns to deterministic latin snake_case -> drop table -> create table -> append rows -> verify.

In [ ]:
import re

import pandas as pd
from pathlib import Path

from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Parameters
csv_path = './datamart_month_excel_enriched_q1.csv'

drp_schema = 'sbx_da'
drp_table = 'tmp_excel_3months_enriched_2026q1'
drp_mode = 'replace'  # replace / append

print('csv_path =', csv_path)
print('target =', f'{drp_schema}.{drp_table}')
print('mode =', drp_mode)

In [ ]:
# Read CSV as object and normalize mixed values safely for DRP write.
df = pd.read_csv(csv_path, dtype=object)

# Optional typed columns.
if 'report_month' in df.columns:
    df['report_month'] = pd.to_datetime(df['report_month'], errors='coerce')

# Normalize text-like columns and convert null markers to Python None.
for c in df.columns:
    if c == 'report_month':
        continue
    s = df[c]
    s = s.where(~s.isna(), None)
    s = s.map(lambda x: str(x).strip() if x is not None else None)
    s = s.replace({'': None, 'None': None, 'nan': None, 'NaN': None, '<NA>': None})
    df[c] = s.astype(object)

# Deterministic latin column mapping (business-friendly).
explicit_map = {
    'report_month': 'report_month',
    'inn_norm': 'inn_norm',
    'contract_number_norm': 'contract_number_norm',
    'ID договора': 'id_dogovora',
    'AUR': 'aur',
    'Амортизация': 'amortization',
    'ВСП договора': 'vsp_dogovora',
    'Всего ТСП': 'vsego_tsp',
    'Дата закрытия договора': 'data_zakrytiya_dogovora',
    'Дата регистрации договора': 'data_registracii_dogovora',
    'Договоры менее 90 дней': 'dogovory_menee_90_dney',
    'ИНН': 'inn_raw',
    'Ко-во торговых точек': 'kolvo_torgovyh_tochek',
    'Код ВСП': 'kod_vsp',
    'Кол-во дней по договору': 'kolvo_dney_po_dogovoru',
    'Кол-во терминалов': 'kolvo_terminalov',
    'Количество операций': 'kolichestvo_operaciy',
    'Количество дней': 'kolichestvo_dney',
    'Комиссия (% с операций)': 'komissiya_proc_ot_operaciy',
    'Комиссия (₽ в месяц)': 'komissiya_rub_v_mesyac',
    'Комиссия МПС (IRF, ₽)': 'komissiya_mps_irf_rub',
    'Комиссия эквайринга': 'komissiya_ekvayringa',
    'Наименование': 'naimenovanie_klienta',
    'Неэффективные ТСП': 'neeffektivnye_tsp',
    'Номер договора': 'nomer_dogovora',
    'Номер организации': 'nomer_organizacii',
    'Отношение к ГК': 'otnoshenie_k_gk',
    'Принадлежность термина': 'prinadlezhnost_termina',
    'Средний IRF %': 'sredniy_irf_proc',
    'Средний экв. %': 'sredniy_ekv_proc',
    'Сумма операций': 'summa_operaciy',
    'Тариф': 'tarif',
    'Учитывать в расчетах': 'uchityvat_v_raschetah',
    'Филиал': 'filial',
    'Фин. Рез.': 'fin_rez',
    'ЧОД': 'chod',
    'Эффективные ТСП': 'effektivnye_tsp',
    'ogrn': 'ogrn',
    'cdi_id': 'cdi_id',
    'ssp_ocrm': 'ssp_ocrm',
}

translit_map = {
    'а':'a','б':'b','в':'v','г':'g','д':'d','е':'e','ё':'e','ж':'zh','з':'z','и':'i','й':'y','к':'k','л':'l','м':'m','н':'n','о':'o','п':'p','р':'r','с':'s','т':'t','у':'u','ф':'f','х':'h','ц':'ts','ч':'ch','ш':'sh','щ':'sch','ъ':'','ы':'y','ь':'','э':'e','ю':'yu','я':'ya'
}

def to_latin_snake(name):
    s = str(name).strip().lower()
    out = []
    for ch in s:
        out.append(translit_map.get(ch, ch))
    s = ''.join(out)
    s = re.sub(r'[^0-9a-zA-Z_]+', '_', s)
    s = re.sub(r'_+', '_', s).strip('_')
    return s

orig_cols = list(df.columns)
safe_cols = []
used = set()
for i, c in enumerate(orig_cols):
    base = explicit_map.get(c, None)
    if base is None:
        base = to_latin_snake(c)
    if base == '' or base[0].isdigit():
        base = f'col_{i+1}'
    name = base
    k = 2
    while name in used:
        name = f'{base}_{k}'
        k += 1
    used.add(name)
    safe_cols.append(name)

col_map_df = pd.DataFrame({'original_column': orig_cols, 'drp_column': safe_cols})
df.columns = safe_cols

print('rows =', len(df))
print('cols =', len(df.columns))
print('column mapping (original -> DRP):')
display(col_map_df)
print('dtypes preview:')
display(df.dtypes.reset_index().rename(columns={'index': 'column', 0: 'dtype'}).head(30))
display(df.head(5))

In [ ]:
# Fill your DRP credentials here and run cell.
drp_conn = connect(
    to='DRP',
    user_params={
        'user_name': 'PUT_YOUR_DRP_LOGIN_HERE',
        'password': 'PUT_YOUR_DRP_PASSWORD_HERE',
    }
)

target_fq = f'{drp_schema}.{drp_table}'

# Robust upload into stable table.
df_upload = df.copy()
for c in df_upload.columns:
    df_upload[c] = df_upload[c].map(lambda x: None if pd.isna(x) else str(x))
    df_upload[c] = df_upload[c].astype(object)

col_defs = []
for c in df_upload.columns:
    col_name = str(c).replace('"', '""')
    col_defs.append(f'"{col_name}" TEXT')

create_sql = f"""
CREATE TABLE {target_fq}
(
  {', '.join(col_defs)}
)
"""

with drp_conn:
    if drp_mode == 'replace':
        drp_conn.execute(f'DROP TABLE IF EXISTS {target_fq}')
    drp_conn.execute(create_sql)
    drp_conn.write(
        table=target_fq,
        df=df_upload,
        mode='append',
    )

print('Upload finished:', target_fq)

In [ ]:
# Verification query
sql_check = f"""
select count(*) as rows_cnt
from {target_fq}
"""

sql_sample = f"""
select *
from {target_fq}
limit 20
"""

with drp_conn:
    cnt_df = drp_conn.fetch(sql_check)
    sample_df = drp_conn.fetch(sql_sample)

print('rows in table:')
display(cnt_df)
print('sample:')
display(sample_df)